# Supplementary Figure — joint intact/word held-out convergence

This figure preserves the original joint intact/word convergence summary after
the main-text Figure 4 was simplified to use an intact-only normalized panel.

The underlying held-out uncertainty calculation matches the corrected analysis:
A→B and B→A are averaged within each outer participant split before summaries
are calculated across outer repetitions.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RUN_CSV = Path(
    "msaa_heldout_topm_outputs_spatial_across/"
    "heldout_topm_crossfit_runs.csv"
)

TOP_M_VALUES = [3, 5, 10]
CONDITIONS = ["intact", "word"]
REPRESENTATIVE_K = 46
T = 300

OUTPUT_DIR = Path("paper_figure_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FIG_PNG = OUTPUT_DIR / "Supplement_joint_intact_word_convergence.png"
FIG_PDF = OUTPUT_DIR / "Supplement_joint_intact_word_convergence.pdf"

if not RUN_CSV.exists():
    raise FileNotFoundError(
        f"Missing {RUN_CSV}. Run revision_04_heldout_topm_msaa_crossfit first."
    )

run_df = pd.read_csv(RUN_CSV)

In [ ]:
def sem(x):
    x = pd.Series(x).dropna()
    if len(x) < 2:
        return np.nan
    return x.std(ddof=1) / np.sqrt(len(x))

outer_df = (
    run_df
    .groupby(
        ["condition", "K", "top_m", "outer_rep"],
        as_index=False,
    )
    .agg(
        accuracy=("mean", "mean"),
    )
)

summary = (
    outer_df
    .groupby(
        ["condition", "K", "top_m"],
        as_index=False,
    )
    .agg(
        mean=("accuracy", "mean"),
        sem=("accuracy", sem),
    )
)

stim = summary[
    summary["condition"].isin(CONDITIONS)
].copy()

stim["relative"] = (
    stim["mean"]
    / stim.groupby(
        ["condition", "top_m"]
    )["mean"].transform("max")
)

joint = (
    stim
    .pivot_table(
        index=["K", "top_m"],
        columns="condition",
        values="relative",
        aggfunc="mean",
    )
    .reset_index()
)

joint["joint_score"] = joint[
    ["intact", "word"]
].min(axis=1)

display(joint.head())

In [ ]:
fig, ax = plt.subplots(figsize=(7.4, 5.0))

styles = {
    3: {"marker": "o", "linestyle": ":",  "label": "top-3"},
    5: {"marker": "s", "linestyle": "--", "label": "top-5"},
    10: {"marker": "^", "linestyle": "-", "label": "top-10"},
}

for top_m in TOP_M_VALUES:
    cur = joint[
        joint["top_m"].astype(int) == top_m
    ].sort_values("K")

    ax.plot(
        cur["K"],
        cur["joint_score"],
        marker=styles[top_m]["marker"],
        linestyle=styles[top_m]["linestyle"],
        linewidth=2.0,
        markersize=5,
        label=styles[top_m]["label"],
    )

ax.axvline(
    REPRESENTATIVE_K,
    color="0.55",
    linestyle="--",
    linewidth=1.2,
)

ax.set_ylim(0, 1.05)
ax.set_xlabel("Model order K", fontsize=12)
ax.set_ylabel("Joint normalized intact/word score", fontsize=12)
ax.set_title(
    "Joint intact/word held-out convergence",
    fontsize=15,
    fontweight="bold",
)
ax.tick_params(labelsize=10)
ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)
ax.legend(frameon=False)

fig.savefig(FIG_PNG, dpi=300, bbox_inches="tight")
fig.savefig(FIG_PDF, bbox_inches="tight")
plt.show()

print("Saved:", FIG_PNG)
print("Saved:", FIG_PDF)